first, trsnalate the maf file using command 
```
~/TOOLS/last-1270/bin/maf-sort -n 2 PipKuh-NycAvi.maf |~/TOOLS/last-1270/bin/maf-swap -n 2 > test.sort.maf
~/TOOLS/last-1270/bin/maf-convert gff -j 50 -J 50 test.sort.maf > test.sort.gff
```

In [2]:
pacman::p_load(circlize, tidyr, dplyr, stringr, readr)

In [3]:
PipKuh.info <- readr::read_tsv('/home/panda2bat/Avivorous_bat/output/09_genome-systeny/N.aviator/lastal/test/PipKuh.info', col_names = c('chr', 'end')) %>% mutate(start = 0) %>% select(c('chr', 'start', 'end')) %>% filter(end > 1000*1000) 
NycAvi.info <- readr::read_tsv('/home/panda2bat/Avivorous_bat/output/09_genome-systeny/N.aviator/lastal/test/NycAvi.info', col_names = c('chr', 'end')) %>% mutate(start = 0) %>% select(c('chr', 'start', 'end')) %>% filter(end > 1000*1000)
PipKuh.info <- PipKuh.info[rev(seq(1, nrow(PipKuh.info))),]
circlize.info <- rbind(NycAvi.info, PipKuh.info)
circlize.info$chr <- factor(circlize.info$chr, levels = c(circlize.info$chr))

Rows: 202 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (1): chr
dbl (1): end

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 194 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (1): chr
dbl (1): end

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
circlize.maf <- as_tibble(ape::read.gff('/home/panda2bat/Avivorous_bat/output/09_genome-systeny/N.aviator/lastz_updated/test.sort.gff')) %>% filter(type == 'match') %>% mutate(attributes = str_split(attributes, ';', simplify = T)[, 2]) %>% mutate(attributes = str_split(attributes, ':', simplify = T)) %>% mutate(target = str_remove(attributes[, 1], 'Name=PipKuh.')) %>% mutate(attributes = str_split(attributes[,2], '-', simplify = T)) %>% mutate(tstart = as.integer(attributes[,1]), tend = as.integer(attributes[,2])) %>% rename('query' = 'seqid', 'qstart' = 'start', 'qend' = 'end') %>% select(c('query', 'qstart', 'qend', 'target', 'tstart', 'tend')) %>% mutate(query = str_remove(query, 'NycAvi.')) %>% filter(query %in% circlize.info$chr) %>% filter(target %in% circlize.info$chr)




In [6]:
pdf('test.pdf')
set1.col = RColorBrewer::brewer.pal(2, 'Set1')
scaffold.col = c('#a6cee3','#1f78b4','#b2df8a','#33a02c','#fb9a99','#e31a1c','#fdbf6f','#ff7f00','#cab2d6','#6a3d9a','#ffff99','#b15928', '#8dd3c7','#bebada','#fdb462','#fccde5','#bc80bd','#984ea3','#7fc97f', '#f0027f')
circos.clear()
circos.par(start.degree = 90, gap.degree = c(rep(1, 19), 10, rep(0, 63), rep(1, 20), 10))
circos.genomicInitialize(circlize.info, plotType = NULL)
circos.track(
	ylim = c(0, 1),
	panel.fun = function(x, y, ...){
		chr = CELL_META$sector.index
		length = CELL_META$xrange
		xmean = mean(CELL_META$xlim)
		num = CELL_META$sector.numeric.index
		if(length >= 20*1000*1000){
			if(str_detect(chr, 'scaffold')){
				circos.text(xmean, CELL_META$ylim[2]+mm_y(3), labels = str_remove(chr, 'scaffold_'), cex = .8, niceFacing = TRUE)	
			} else {
				circos.text(xmean, CELL_META$ylim[2]+mm_y(3), labels = as.character(105-num), cex = .8, niceFacing = TRUE)
			}
		}
	},
	track.height = .03,
	cell.padding = c(0, 1, 0.02, 1)
)
highlight.chromosome(NycAvi.info$chr, col = set1.col[1], track.index = 1)
highlight.chromosome(PipKuh.info$chr, col = set1.col[2], track.inddex = 1)
circos.genomicTrack(
	ylim = c(0, 1), 
	panel.fun = function(x, y, ...){
		chr = CELL_META$sector.index
		length = CELL_META$xrange
		xlim = CELL_META$xlim
		ylim = CELL_META$ylim
		num = CELL_META$sector.numeric.index
		if(length >= 20*1000*1000){
			if(str_detect(chr, 'scaffold')){
				circos.rect(xlim[1], 0, xlim[2], 1, col = scaffold.col[num], border = NA)
			} else {
				circos.rect(xlim[1], 0, xlim[2], 1, col = 'white', border = 'black')
			}
		} else {
			circos.rect(xlim[1], 0, xlim[2], 1, col = 'gray', border = NA)
		}
	},
	track.height = .08,
	bg.border = NA,
	cell.padding = c(.02, 0, .02, 0)
)
for(i in seq(1, 20)){
	chr = paste0('scaffold_', i)
	bed1 <- circlize.maf %>% filter(query == chr) %>% select(c(1:3))
	bed2 <- circlize.maf %>% filter(query == chr) %>% select(c(4:6))
	col = scaffold.col[i]
	circos.genomicLink(bed1, bed2, col = col, border = NA)
}
dev.off()


Warning message in RColorBrewer::brewer.pal(2, "Set1"):
“minimal value for n is 3, returning requested palette with 3 different levels
”
Note: 1 point is out of plotting region in sector 'scaffold_1', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_1', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_2', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_2', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_3', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_3', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_4', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_4', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_5', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_5', track
'1'.

Note: 1 point is out of plotting region in sector 'scaffold_6', track
'1'.

Note: 1 point is out of plo

png 
  2